In [103]:
!python3 -V

Python 3.11.7


In [19]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pickle
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge
from sklearn.metrics import root_mean_squared_error

In [3]:
def read_dataFrame(filePath):
    df = pd.read_parquet(filePath)
    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)
    
    # It makes more sense to focus on trips which took between (1-60 miniutes)
    df = df[(df.duration >= 1) & (df.duration <= 60)]

    #Combining Categorical Feature
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    return df
    
    

In [4]:
#Read data
df_train = read_dataFrame('./data/green_tripdata_2023-01.parquet')
df_val = read_dataFrame('./data/green_tripdata_2023-02.parquet')

In [21]:
#Feature Engineering
#Transforming datetime to Extract Date Components & Cyclical Features for Time Components on Training Dataset 
df_train['day_of_week'] = df_train['lpep_pickup_datetime'].dt.dayofweek
df_train['day'] = df_train['lpep_pickup_datetime'].dt.day
df_train['hour'] = df_train['lpep_pickup_datetime'].dt.hour
df_train['hour_sin'] = np.sin(2 * np.pi * df_train['hour'] / 24)
df_train['hour_cos'] = np.cos(2 * np.pi * df_train['hour'] / 24)
df_train['day_of_week_sin'] = np.sin(2 * np.pi * df_train['day_of_week'] / 7)
df_train['day_of_week_cos'] = np.cos(2 * np.pi * df_train['day_of_week'] / 7)

#Transforming datetime to Extract Date Components & Cyclical Features for Time Components on Valuation Dataset 
df_val['day_of_week'] = df_val['lpep_pickup_datetime'].dt.dayofweek
df_val['day'] = df_val['lpep_pickup_datetime'].dt.day
df_val['hour'] = df_val['lpep_pickup_datetime'].dt.hour
df_val['hour_sin'] = np.sin(2 * np.pi * df_val['hour'] / 24)
df_val['hour_cos'] = np.cos(2 * np.pi * df_val['hour'] / 24)
df_val['day_of_week_sin'] = np.sin(2 * np.pi * df_val['day_of_week'] / 7)
df_val['day_of_week_cos'] = np.cos(2 * np.pi * df_val['day_of_week'] / 7)


In [39]:
#Processing data
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
# numerical = ['trip_distance', 'day', 'day_of_week', 'hour', 'hour_sin', 'hour_cos', 'day_of_week_sin', 'day_of_week_cos']
numerical = ['trip_distance', 'day_of_week', 'hour_sin', 'hour_cos', 'day_of_week_sin', 'day_of_week_cos']


dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [36]:
# Q5. Training a base model (Simple LinearRegression Model)
# Now let's use the feature matrix from the previous step to train a model.
# Train a plain linear regression model with default parameters
# Calculate the RMSE of the model on the training data
target = 'duration'
y_train = df_train[target].values
y_val = df_train[target].values
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_train)
root_mean_squared_error(y_val, y_pred)


4.784067610854723

In [37]:
# Q6. Evaluating the model on validation dataset (February 2023)

# Now let's apply this model to the validation dataset (February 2023).
# What's the RMSE on validation?
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

# Training the linear regression model
lr = LinearRegression()
lr.fit(X_train, y_train)

#Prediction & Evaluating the model
y_pred = lr.predict(X_val)
root_mean_squared_error(y_val, y_pred)

6.030914807846822

In [31]:
#Training the lasso regression model
rr = Lasso(0.01)
rr.fit(X_train, y_train)
y_pred = rr.predict(X_val)
root_mean_squared_error(y_val, y_pred)

8.145856901325871

In [141]:
#Saving the best model i.e Ridge Model
with open('models/riged_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)